<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [1]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

In [2]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.0 MB/s eta 0:00:00


### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [3]:
import io

In [108]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [109]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"])).convert("RGB")
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        labels = [label - 1 for label in labels]

        # Convert COCO [x, y, w, h] to Pascal VOC [x1, y1, x2, y2]
        boxes = np.array(row['bbox'].tolist())
        if len(boxes) > 0:
            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [110]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])
)

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()


Не забываем инициализировать наш датасет

In [111]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [112]:
import torchvision.models as models
import torch.nn as nn

In [113]:
class Backbone(nn.Module):
    def __init__(self, name='resnet50', pretrained=True, unfreeze_last_k_stages=0):
        super().__init__()
        if name == 'resnet50':
            self.model = models.resnet50(pretrained=pretrained)
        else:
            raise ValueError(f"Backbone {name} not supported.")

        # Freeze all parameters initially
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze the last k stages
        # ResNet stages are layer1, layer2, layer3, layer4. Unfreezing "last k" means layer4, then layer3, etc.
        stages_to_unfreeze_modules = [
            self.model.layer4,  # Last stage
            self.model.layer3,
            self.model.layer2,
            self.model.layer1   # First stage
        ]

        for i in range(min(unfreeze_last_k_stages, len(stages_to_unfreeze_modules))):
            current_stage_module = stages_to_unfreeze_modules[i]
            for param in current_stage_module.parameters():
                param.requires_grad = True

    def forward(self, x):
        # Initial layers
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        # Feature maps from each stage (C1, C2, C3, C4)
        c1 = self.model.layer1(x)
        c2 = self.model.layer2(c1)
        c3 = self.model.layer3(c2)
        c4 = self.model.layer4(c3)

        # Return features from C1, C2, C3, C4, ordered from lowest to highest resolution
        return [c1, c2, c3, c4]

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [114]:
import torch.nn.functional as F

class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels):
        super().__init__()
        self.in_channels_list = in_channels_list
        self.out_channels = out_channels

        # Lateral convolutions to reduce channel dimensionality to out_channels
        self.lateral_convs = nn.ModuleList()
        for in_channels in in_channels_list:
            self.lateral_convs.append(
                nn.Conv2d(in_channels, out_channels, kernel_size=1)
            )

        # Smooth convolutions for each FPN output level (P_i)
        # These are applied after the sum with upsampled feature
        self.fpn_convs = nn.ModuleList()
        for _ in range(len(in_channels_list)): # One for each FPN level
            self.fpn_convs.append(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            )

    def forward(self, features):
        """
        Args:
            features (list[Tensor]): Feature maps from the backbone,
                                     ordered from lowest to highest resolution (e.g., C1, C2, C3, C4).
                                     For ResNet50, this would typically be outputs of layer1, layer2, layer3, layer4.
        Returns:
            list[Tensor]: FPN feature maps (P1, P2, P3, P4), ordered from lowest to highest resolution.
                          (P_i corresponds to C_i resolution)
        """
        num_levels = len(features)

        # Apply lateral convolutions to get features with uniform channels
        # lateral_features = [L_C1, L_C2, L_C3, L_C4]
        lateral_features = [
            self.lateral_convs[i](features[i])
            for i in range(num_levels)
        ]

        # Initialize FPN outputs list
        fpn_outputs = [None] * num_levels

        # Start from the highest resolution level (last feature in the list)
        # e.g., for features=[C1, C2, C3, C4], this is C4 (index 3)
        # FPN output P4 is generated from C4, optionally smoothed
        fpn_outputs[num_levels - 1] = self.fpn_convs[num_levels - 1](lateral_features[num_levels - 1])

        # Iterate down from (num_levels - 2) to 0 (e.g., C3, C2, C1)
        for i in range(num_levels - 2, -1, -1):
            # Upsample the higher-resolution FPN feature (e.g., P4 for C3 -> P3)
            # The size should match the current lateral feature (e.g., L_C3)
            upsampled_feature = F.interpolate(
                fpn_outputs[i + 1],  # P_{i+1}
                size=lateral_features[i].shape[2:], # Target size is the spatial size of C_i
                mode='nearest'
            )
            # Add with the lateral feature of the current level (L_Ci)
            fused_feature = lateral_features[i] + upsampled_feature
            # Apply the smoothing convolution to get the final FPN output for this level (P_i)
            fpn_outputs[i] = self.fpn_convs[i](fused_feature)

        return fpn_outputs # Returns [P1, P2, P3, P4] (e.g., P2, P3, P4, P5 for a 4-level FPN)

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [115]:
import torch.nn as nn

class Head(nn.Module):
    def __init__(self, in_channels, num_classes, num_convs=4):
        super().__init__()
        self.num_classes = num_classes

        # Classification branch
        cls_branch = []
        for _ in range(num_convs):
            cls_branch.append(nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1))
            cls_branch.append(nn.ReLU(inplace=True))
        self.cls_branch = nn.Sequential(*cls_branch)
        self.cls_output = nn.Conv2d(in_channels, num_classes, kernel_size=1)

        # Regression branch
        reg_branch = []
        for _ in range(num_convs):
            reg_branch.append(nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1))
            reg_branch.append(nn.ReLU(inplace=True))
        self.reg_branch = nn.Sequential(*reg_branch)
        # 4 for bbox coordinates (e.g., dx, dy, dw, dh or x1, y1, x2, y2)
        self.reg_output = nn.Conv2d(in_channels, 4, kernel_size=1)

        # Objectness branch (confidence score)
        obj_branch = []
        for _ in range(num_convs):
            obj_branch.append(nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1))
            obj_branch.append(nn.ReLU(inplace=True))
        self.obj_branch = nn.Sequential(*obj_branch)
        self.obj_output = nn.Conv2d(in_channels, 1, kernel_size=1) # 1 for objectness score

    def forward(self, features):
        """
        Args:
            features (list[Tensor]): FPN feature maps (P1, P2, P3, P4), ordered from lowest to highest resolution.
        Returns:
            tuple: (cls_preds, reg_preds, obj_preds)
                   cls_preds (list[Tensor]): List of class predictions for each FPN level.
                   reg_preds (list[Tensor]): List of bounding box regression predictions for each FPN level.
                   obj_preds (list[Tensor]): List of objectness predictions for each FPN level.
        """
        cls_preds = []
        reg_preds = []
        obj_preds = []

        for feature_map in features:
            cls_feat = self.cls_branch(feature_map)
            cls_pred = self.cls_output(cls_feat)
            cls_preds.append(cls_pred)

            reg_feat = self.reg_branch(feature_map)
            reg_pred = self.reg_output(reg_feat)
            reg_preds.append(reg_pred)

            obj_feat = self.obj_branch(feature_map)
            obj_pred = self.obj_output(obj_feat)
            obj_preds.append(obj_pred)

        return cls_preds, reg_preds, obj_preds

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [116]:
class Detector(nn.Module):
    def __init__(self, num_classes, backbone_name='resnet50', pretrained=True, unfreeze_last_k_stages=0, fpn_out_channels=256, head_num_convs=4):
        super().__init__()
        self.backbone = Backbone(name=backbone_name, pretrained=pretrained, unfreeze_last_k_stages=unfreeze_last_k_stages)

        # Assuming ResNet50 output channels for layer1, layer2, layer3, layer4
        backbone_out_channels = [256, 512, 1024, 2048]
        self.neck = Neck(in_channels_list=backbone_out_channels, out_channels=fpn_out_channels)

        self.head = Head(in_channels=fpn_out_channels, num_classes=num_classes, num_convs=head_num_convs)

    def forward(self, x):
        # Pass through backbone
        c_features = self.backbone(x)

        # Pass through neck (FPN)
        fpn_features = self.neck(c_features)

        # Pass through head to get predictions
        cls_preds, reg_preds, obj_preds = self.head(fpn_features)

        return cls_preds, reg_preds, obj_preds

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [136]:
import torch

def iou_calc(boxes1, boxes2):
    x1 = torch.max(boxes1[:, 0].unsqueeze(1), boxes2[:, 0].unsqueeze(0))
    y1 = torch.max(boxes1[:, 1].unsqueeze(1), boxes2[:, 1].unsqueeze(0))
    x2 = torch.min(boxes1[:, 2].unsqueeze(1), boxes2[:, 2].unsqueeze(0))
    y2 = torch.min(boxes1[:, 3].unsqueeze(1), boxes2[:, 3].unsqueeze(0))

    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    union = area1.unsqueeze(1) + area2.unsqueeze(0) - intersection
    return intersection / (union + 1e-7)

def TAL_assigner(pred_boxes, gt_boxes, pred_cls_scores, gt_labels, anchor_points,
                 alpha=1.0, beta=6.0, topk_candidates=10):
    num_gts = gt_boxes.shape[0]
    num_preds = pred_boxes.shape[0]

    if num_gts == 0 or num_preds == 0:
        return (
            torch.zeros((num_preds, 4), device=pred_boxes.device),
            torch.full((num_preds,), -1, dtype=torch.long, device=pred_boxes.device),
            torch.zeros((num_preds,), dtype=torch.bool, device=pred_boxes.device)
        )

    ious = iou_calc(pred_boxes, gt_boxes) # (num_preds, num_gts)

    # Берем скоры только для нужных GT классов
    # pred_cls_scores: (num_preds, num_classes)
    cls_scores = pred_cls_scores[:, gt_labels] # (num_preds, num_gts)

    # Alignment metric t = s^alpha * u^beta
    # Добавляем небольшой эпсилон перед возведением в степень для стабильности
    alignment_metric = (cls_scores.clamp(min=1e-9).pow(alpha) * ious.pow(beta))

    # Фильтр: центр якоря должен быть внутри GT
    anchor_points_x = anchor_points[:, 0].unsqueeze(1)
    anchor_points_y = anchor_points[:, 1].unsqueeze(1)

    is_in_gt = (
        (anchor_points_x >= gt_boxes[:, 0].unsqueeze(0)) & (anchor_points_x <= gt_boxes[:, 2].unsqueeze(0)) &
        (anchor_points_y >= gt_boxes[:, 1].unsqueeze(0)) & (anchor_points_y <= gt_boxes[:, 3].unsqueeze(0))
    )

    alignment_metric = alignment_metric * is_in_gt.float()

    # Выбираем top-k кандидатов для каждого GT
    _, topk_indices = torch.topk(alignment_metric, k=min(topk_candidates, num_preds), dim=0)
    mask_topk = torch.zeros_like(alignment_metric, dtype=torch.bool)
    mask_topk.scatter_(0, topk_indices, True)

    alignment_metric = alignment_metric * mask_topk.float()

    # Если один предсказание подходит нескольким GT, берем тот, где выше IoU
    best_gt_alignment, best_gt_idx = alignment_metric.max(dim=1)
    pos_mask = best_gt_alignment > 1e-9

    assigned_gt_labels = torch.full((num_preds,), -1, dtype=torch.long, device=pred_boxes.device)
    assigned_gt_boxes = torch.zeros((num_preds, 4), device=pred_boxes.device)

    if pos_mask.any():
        assigned_gt_labels[pos_mask] = gt_labels[best_gt_idx[pos_mask]]
        assigned_gt_boxes[pos_mask] = gt_boxes[best_gt_idx[pos_mask]]

    return assigned_gt_boxes, assigned_gt_labels, pos_mask

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [118]:
from torchvision.ops import distance_box_iou_loss

In [119]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [120]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [121]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.0264567136764526


In [122]:
def diou_loss(pred_boxes, gt_boxes):
    # 1. IoU calculation with stability
    x1 = torch.max(pred_boxes[:, 0], gt_boxes[:, 0])
    y1 = torch.max(pred_boxes[:, 1], gt_boxes[:, 1])
    x2 = torch.min(pred_boxes[:, 2], gt_boxes[:, 2])
    y2 = torch.min(pred_boxes[:, 3], gt_boxes[:, 3])

    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area_pred = (pred_boxes[:, 2] - pred_boxes[:, 0]).clamp(0) * (pred_boxes[:, 3] - pred_boxes[:, 1]).clamp(0)
    area_gt = (gt_boxes[:, 2] - gt_boxes[:, 0]).clamp(0) * (gt_boxes[:, 3] - gt_boxes[:, 1]).clamp(0)
    union = area_pred + area_gt - intersection + 1e-7
    iou = (intersection / union).clamp(0, 1)

    # 2. Distance between centers
    p_cx, p_cy = (pred_boxes[:, 0] + pred_boxes[:, 2]) / 2, (pred_boxes[:, 1] + pred_boxes[:, 3]) / 2
    g_cx, g_cy = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2, (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
    d2 = (p_cx - g_cx)**2 + (p_cy - g_cy)**2

    # 3. Smallest enclosing box diagonal
    c_x1 = torch.min(pred_boxes[:, 0], gt_boxes[:, 0])
    c_y1 = torch.min(pred_boxes[:, 1], gt_boxes[:, 1])
    c_x2 = torch.max(pred_boxes[:, 2], gt_boxes[:, 2])
    c_y2 = torch.max(pred_boxes[:, 3], gt_boxes[:, 3])
    c2 = (c_x2 - c_x1)**2 + (c_y2 - c_y1)**2

    # DIoU = 1 - IoU + (d^2 / c^2)
    # The distance penalty is also clamped for stability
    diou_val = 1 - iou + (d2 / (c2 + 1e-7)).clamp(0, 1)
    return diou_val.mean()

In [123]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [143]:
import torch
import gc
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

# 1. Очистка памяти
torch.cuda.empty_cache()
gc.collect()

# 2. Уменьшаем размер батча для экономии памяти
NEW_BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=NEW_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(test_dataset, batch_size=NEW_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# 3. Настройка обучения
num_epochs_final = 15
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs_final)

print(f"Starting final training sprint with Batch Size {NEW_BATCH_SIZE} for {num_epochs_final} epochs...")

for epoch in range(num_epochs_final):
    model.train()
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs_final}")

    for images, targets in pbar:
        images = images.to(device)
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        cls_preds, reg_preds, obj_preds = model(images)
        loss = criterion(cls_preds, reg_preds, obj_preds, targets)

        if torch.isnan(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{optimizer.param_groups[0]['lr']:.6f}"})

    scheduler.step()

    # Валидация каждые 5 эпох для мониторинга
    if (epoch + 1) % 5 == 0:
        mAP_check = validate(dataloader=val_loader, filter_predictions_func=filter_predictions, device=device, strides=STRIDES, score_threshold=0.05)
        print(f"Epoch {epoch+1} Intermediate mAP: {mAP_check:.4f}")

# 4. Финальная проверка mAP
print("Calculating final mAP on test set...")
final_test_mAP = validate(dataloader=val_loader, filter_predictions_func=filter_predictions, device=device, strides=STRIDES, score_threshold=0.05)
print(f"\nFINAL TEST mAP: {final_test_mAP:.4f}")

"""
### Ответы на вопросы:

1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
TAL (Task Alignment Learning). Он динамически подбирает веса для классификации и регрессии, позволяя модели фокусироваться на тех якорях, которые одновременно хорошо классифицируются и точно описывают границы объекта.

2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
Feature Pyramid Network (FPN). Объединение семантически богатых признаков глубоких слоев с пространственно точными признаками ранних слоев критично для обнаружения мелких объектов в Halo (шлемы, оружие).

3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?
Использование DIoU лосса на самых первых итерациях. Пока классификатор не научился отличать объект от фона, точность локализации (IoU) не вносит вклада в mAP, так как предсказания не проходят порог уверенности.
"""

Starting final training sprint with Batch Size 4 for 15 epochs...


Epoch 1/15:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 2/15:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 3/15:   0%|          | 0/116 [00:00<?, ?it/s]

Epoch 4/15:   0%|          | 0/116 [00:00<?, ?it/s]

...

Running validation:   0%|          | 0/34 [00:00<?, ?it/s]

Epoch 15 Intermediate mAP: 0.0000
Calculating final mAP on test set...


Running validation:   0%|          | 0/34 [00:00<?, ?it/s]


FINAL TEST mAP: 0.0000


"\n### Ответы на вопросы:\n\n1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?\nTAL (Task Alignment Learning). Он динамически подбирает веса для классификации и регрессии, позволяя модели фокусироваться на тех якорях, которые одновременно хорошо классифицируются и точно описывают границы объекта.\n\n2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?\nFeature Pyramid Network (FPN). Объединение семантически богатых признаков глубоких слоев с пространственно точными признаками ранних слоев критично для обнаружения мелких объектов в Halo (шлемы, оружие).\n\n3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?\nИспользование DIoU лосса на самых первых итерациях. Пока классификатор не научился отличать объект от фона, точность локализации (IoU) не вносит вклада в mAP, так как предсказания не проходят порог уверенности.\n"

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [141]:
from torchmetrics.detection import MeanAveragePrecision
from tqdm.auto import tqdm

@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    model.eval()
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)

        # Приводим к формату для torchmetrics и перемещаем на CPU
        targets_cpu = [{k: v.to('cpu') if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
        predicts_cpu = [{k: v.to('cpu') if isinstance(v, torch.Tensor) else v for k, v in p.items()} for p in predicts]

        metric.update(predicts_cpu, targets_cpu)
    return metric.compute()["map"].item()